# QIAS RAG - Minimal Colab (HF / Unsloth)

Minimal Google Colab workflow for end-to-end smoke testing.

- Backend: `huggingface` (default) or `unsloth` (optional GPU path)
- No Ollama usage in this notebook


In [ ]:
# 1) GPU check
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 2) Mount Drive and select backend
from google.colab import drive
drive.mount('/content/drive')

CLIENT_TYPE = "huggingface"  # choose: "huggingface" or "unsloth"
assert CLIENT_TYPE in {"huggingface", "unsloth"}
print("Selected backend:", CLIENT_TYPE)

REPO_URL = "https://github.com/swaileh/qias-mawarith-rag.git"
REPO_DIR = "/content/qias_rag_thinking"
SHARED_TASK_DIR = "/content/qias_shared_task_2026"
PDF_DIR = "/content/drive/MyDrive/QIAS26/qias_rag_thinking/data/pdfs"


In [ ]:
# 3) Clone/update repos
import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if not os.path.exists(SHARED_TASK_DIR):
    !git clone https://github.com/QIAS/qias_shared_task_2026.git {SHARED_TASK_DIR}
else:
    !git -C {SHARED_TASK_DIR} pull

%cd /content/qias_rag_thinking


In [ ]:
# 4) Install dependencies
!pip install -q -U pip
!pip install -q -e ".[dev]"
!pip install -q bitsandbytes accelerate

if CLIENT_TYPE == "unsloth":
    !pip install -q unsloth
    print("Installed Unsloth dependencies")
else:
    print("Using HuggingFace dependencies")


In [ ]:
# 5) Configure backend and init pipeline
import sys
import yaml
from pathlib import Path

config_path = Path("/content/qias_rag_thinking/config/rag_config.yaml")
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

config["model"]["client_type"] = CLIENT_TYPE
config["model"]["max_new_tokens"] = 1024
config["model"]["context_window"] = 8192
config.setdefault("evaluation", {})["enable_relevance_evaluation"] = False

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

sys.path.insert(0, "/content/qias_rag_thinking")
from qias_mawarith_rag.pipeline import RAGPipeline

pipeline = RAGPipeline(config_path=str(config_path))
print("Pipeline initialized with backend:", pipeline.config["model"]["client_type"])


In [ ]:
# 6) Build/load knowledge base
import os

if os.path.exists(PDF_DIR):
    pdf_files = [f for f in os.listdir(PDF_DIR) if f.lower().endswith('.pdf')]
    print("PDF files found:", len(pdf_files))
    if pdf_files:
        pipeline.build_knowledge_base(PDF_DIR)
        stats = pipeline.vector_store.get_collection_stats()
        print("KB docs:", stats.get("total_documents", 0))
    else:
        print("No PDFs found. Upload PDFs to:", PDF_DIR)
else:
    print("PDF directory missing:", PDF_DIR)


In [ ]:
# 7) One-query latency smoke benchmark
import time

question = "مات وترك: أم و أب و بنت. ما هو نصيب كل وريث؟"
t0 = time.perf_counter()
result = pipeline.query(question, top_k=5)
latency_s = time.perf_counter() - t0

print(f"Latency: {latency_s:.2f}s")
print("Parsing success:", result.get("parsed_output", {}).get("parsing_success"))
print("Validation success:", result.get("parsed_output", {}).get("validation_success"))
print("Retrieved docs:", len(result.get("retrieved_docs", [])))
